Cài đặt môi trường

In [ ]:
import os
import sys
import shutil
import importlib.util

HOME = '/kaggle/working'
os.chdir(HOME)

print("🚀 BẮT ĐẦU KHỞI TẠO MÔI TRƯỜNG TRÊN KAGGLE...")

# --- BƯỚC 1: CHUẨN BỊ THƯ MỤC CHECKPOINT ---
MODEL_DIR = os.path.join(HOME, 'checkpoints')
os.makedirs(MODEL_DIR, exist_ok=True)
checkpoint_file = os.path.join(MODEL_DIR, 'oriented_rcnn_r50_fpn_1x_dota_le90-6d2b2ce0.pth')

# --- BƯỚC 2: CÀI ĐẶT THƯ VIỆN (QUAN TRỌNG) ---
print("\n⬇️ [2/5] Cài đặt thư viện nền (Binary)...")

# Gỡ các bản cũ có sẵn của Kaggle để tránh xung đột
!pip uninstall -y mmcv mmcv-full mmdet mmrotate > /dev/null 2>&1

!pip install torch==2.3.0 torchvision==0.18.0 --index-url https://download.pytorch.org/whl/cu121 > /dev/null
!pip install mmcv==2.2.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.3/index.html > /dev/null
!pip install mmengine "mmdet>=3.0.0" > /dev/null

# --- BƯỚC 3: TẢI MÃ NGUỒN TỪ GITHUB ---
print("\n⬇️ [3/5] Tải mã nguồn MMRotate mới nhất...")
MMROTATE_DIR = os.path.join(HOME, 'mmrotate')

if os.path.exists(MMROTATE_DIR):
    shutil.rmtree(MMROTATE_DIR)

!git clone -b dev-1.x https://github.com/open-mmlab/mmrotate.git {MMROTATE_DIR}

# --- BƯỚC 4: VÁ LỖI TỰ ĐỘNG ---
print("\n🔧 [4/5] Vá lỗi hệ thống...")

# 4.1. Tạo version.py
version_file = os.path.join(MMROTATE_DIR, 'mmrotate/version.py')
with open(version_file, 'w') as f:
    f.write("__version__ = '1.0.0rc1'\nshort_version = '1.0.0'\nversion_info = (1, 0, 0, 'rc1')")

# 4.2. Sửa MMDet
det_spec = importlib.util.find_spec("mmdet")
if det_spec and det_spec.origin:
    det_init = det_spec.origin
    print(f"   -> Đã tìm thấy MMDet tại: {det_init}")
    with open(det_init, 'w') as f:
        f.write("# Copyright (c) OpenMMLab. All rights reserved.\nimport mmcv\nimport mmengine\nfrom .version import __version__, version_info\n__all__ = ['__version__', 'version_info']")
else:
    print("   ⚠️ CẢNH BÁO: Không tìm thấy mmdet để vá lỗi!")

# 4.3. Sửa MMRotate __init__.py
rot_init = os.path.join(MMROTATE_DIR, 'mmrotate/__init__.py')
safe_rot = """
import mmcv
import mmdet
from .version import __version__, version_info
from mmengine.utils import digit_version
mmcv_version = digit_version(mmcv.__version__)
__all__ = ['__version__', 'version_info', 'digit_version', 'mmcv_version']
"""
with open(rot_init, 'w') as f:
    f.write(safe_rot)

# Cài đặt code vừa tải
%cd {MMROTATE_DIR}
!pip install -v -e . > /dev/null

# --- BƯỚC 5: TẢI MODEL ---
print(f"\n💾 [5/5] Kiểm tra Model: {checkpoint_file}")
# Quay về thư mục làm việc chính
os.chdir(HOME)

if os.path.exists(checkpoint_file) and os.path.getsize(checkpoint_file) > 100000000:
    print("   ✅ Đã có Model! Sẵn sàng sử dụng.")
else:
    print("   ⬇️ Đang tải Model từ HuggingFace (thay thế Drive)...")
    url = "https://huggingface.co/dl4eo/Oriented_R-CNN_pretrained_on_DOTA_1.0/resolve/main/oriented_rcnn_r50_fpn_1x_dota_le90-6d2b2ce0.pth?download=true"
    # Dùng curl trên Kaggle rất nhanh
    !curl -L "{url}" -o "{checkpoint_file}"
    print("   ✅ Đã tải xong!")

# --- HOÀN TẤT ---
print("\n" + "="*50)
print("🎉 MÔI TRƯỜNG KAGGLE ĐÃ SẴN SÀNG!")
print(f"✅ Code: {MMROTATE_DIR}")
print(f"✅ Model: {checkpoint_file}")
print("="*50)

try:
    import mmrotate
    print("Test Import mmrotate: THÀNH CÔNG ✅")
except ImportError as e:
    print(f"Test Import mmrotate: THẤT BẠI ❌ - Lỗi: {e}")

In [1]:
# CELL 2: IMPORT ALL
import sys, os, warnings, subprocess

# --- FIX COMPATIBILITY (NUMPY & PROTOBUF) ---
def fix_environment():
    needs_restart = False
    try:
        import numpy as np
        if int(np.__version__.split('.')[0]) >= 2:
            print(f"⚠ Phát hiện NumPy {np.__version__}. Đang hạ cấp xuống 1.x...")
            subprocess.run(["pip", "install", "numpy<2.0.0", "-q"])
            needs_restart = True
    except ImportError:
        subprocess.run(["pip", "install", "numpy<2.0.0", "-q"])
        needs_restart = True

    try:
        import google.protobuf
        # Kiểm tra version, nếu đầu 4 hoặc 5 thì hạ cấp
        if not google.protobuf.__version__.startswith("3.20"):
            print(f"⚠ Phát hiện Protobuf {google.protobuf.__version__}. Đang hạ cấp xuống 3.20.3...")
            subprocess.run(["pip", "install", "protobuf==3.20.3", "-q"])
            needs_restart = True
    except ImportError:
        subprocess.run(["pip", "install", "protobuf==3.20.3", "-q"])
        needs_restart = True

    if needs_restart:
        print("✅ Đã cài đặt xong các bản vá lỗi!")
        print("🔴 QUAN TRỌNG: BẠN CẦN RESTART SESSION NGAY LẬP TỨC.")
        print("👉 (Click vào menu 'Run' -> 'Restart Session', sau đó chạy lại Cell này)")
        sys.exit("Dừng chương trình để Restart Session.")

fix_environment()

import glob, json, random, time, shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch

# Tắt các cảnh báo đỏ
warnings.filterwarnings("ignore")

# OpenMMLab Core
import mmcv
import mmengine
import mmdet
import mmrotate

# Modules cho Training & Config
from mmengine.config import Config
from mmengine.runner import Runner

# Modules cho Inference & Demo
from mmdet.apis import init_detector, inference_detector
from mmrotate.registry import VISUALIZERS

# Gradio cho Demo App
try:
    import gradio as gr
except ImportError:
    print("Đang cài đặt Gradio...")
    subprocess.run(["pip", "install", "gradio", "-q"])
    import gradio as gr

print(f"✅ Hoàn tất Import.")
print(f"- NumPy version: {np.__version__}")
try:
    import google.protobuf
    print(f"- Protobuf version: {google.protobuf.__version__}")
except:
    pass

2025-12-19 14:25:30.266448: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766154330.440405     216 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766154330.487754     216 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


✅ Hoàn tất Import.
- NumPy version: 1.26.4
- Protobuf version: 3.20.3


KIỂM TRA DỮ LIỆU (VISUALIZATION)

In [ ]:
DATASET_DIR = '/kaggle/input/processed-data/processed_mmrotate_data/' 

found_train_dirs = glob.glob(os.path.join(DATASET_DIR, '**', 'train', 'images'), recursive=True)

if not found_train_dirs:
    print("❌ LỖI: Không tìm thấy thư mục 'train/images' trong /kaggle/input")
    print("👉 Hãy chắc chắn bạn đã Add Dataset vào Notebook này.")
    # Fallback (dự phòng) để không crash nếu chưa add data
    TRAIN_IMG = ""
    TRAIN_LBL = ""
else:
    # Lấy đường dẫn cha của 'images' làm DATA_ROOT
    img_dir_path = found_train_dirs[0] 
    train_dir = os.path.dirname(img_dir_path) # .../train
    DATA_ROOT = os.path.dirname(train_dir)    # .../processed_mmrotate_data
    
    TRAIN_IMG = img_dir_path
    TRAIN_LBL = os.path.join(train_dir, 'labelTxt')
    
    print(f"✅ Đã tìm thấy dữ liệu tại: {DATA_ROOT}")
    print(f"   📂 Folder ảnh: {TRAIN_IMG}")
    print(f"   📂 Folder nhãn: {TRAIN_LBL}")


# --- HÀM VISUALIZE ---
def visualize_sample(filename):
    if not TRAIN_IMG: return

    # Tìm file ảnh (hỗ trợ cả png và jpg)
    img_path = os.path.join(TRAIN_IMG, filename + '.png')
    if not os.path.exists(img_path):
        img_path = os.path.join(TRAIN_IMG, filename + '.jpg')

    txt_path = os.path.join(TRAIN_LBL, filename + '.txt')

    if os.path.exists(img_path) and os.path.exists(txt_path):
        img = cv2.imread(img_path)
        if img is None: 
            print(f"⚠️ Không đọc được ảnh: {img_path}")
            return
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        with open(txt_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 8:
                    coords = list(map(float, parts[:8]))
                    poly = np.array(coords, dtype=np.int32).reshape((-1, 1, 2))
                    
                    # Vẽ đa giác (Polygon) màu xanh lá
                    cv2.polylines(img, [poly], isClosed=True, color=(0, 255, 0), thickness=2)
                    
                    # Vẽ điểm đầu (để biết chiều xoay) màu đỏ
                    cv2.circle(img, (int(coords[0]), int(coords[1])), 5, (255, 0, 0), -1)

        plt.figure(figsize=(10, 10))
        plt.imshow(img)
        plt.title(f"Check Data: {filename}")
        plt.axis('off')
        plt.show()
    else:
        print(f"⚠️ Thiếu file ảnh hoặc file txt cho: {filename}")

# --- CHẠY KIỂM TRA ---
found_files = []
if TRAIN_IMG and os.path.exists(TRAIN_IMG):
    with os.scandir(TRAIN_IMG) as entries:
        for entry in entries:
            if entry.is_file() and entry.name.lower().endswith(('.png', '.jpg', '.jpeg')):
                found_files.append(entry.name)

            if len(found_files) >= 100:
                break

    if found_files:
        num_samples = min(5, len(found_files))
        random_files = random.sample(found_files, num_samples)

        print(f"\n📸 Đã chọn ngẫu nhiên {num_samples} ảnh để kiểm tra:")
        for file_name in random_files:
            sample_name = os.path.splitext(file_name)[0]
            visualize_sample(sample_name)
    else:
        print("❌ Thư mục ảnh rỗng!")
else:
    if not TRAIN_IMG:
        print("❌ Chưa cấu hình được đường dẫn dữ liệu.")
    else:
        print(f"❌ Đường dẫn không tồn tại: {TRAIN_IMG}")

CẤU HÌNH & HUẤN LUYỆN (TRAINING)

In [ ]:
# 1. Tự động tìm đường dẫn Data Root
found_train = glob.glob('/kaggle/input/**/train/images', recursive=True)
if found_train:
    detected_data_root = os.path.dirname(os.path.dirname(found_train[0])) + '/'
    print(f"✅ Đã tìm thấy Data Root: {detected_data_root}")
else:
    detected_data_root = '/kaggle/input/processed-data/processed_mmrotate_data/'
    print("⚠️ Không tìm thấy data tự động. Bạn hãy thay thủ công đường dẫn bên dưới.")

# 2. Tự động tìm file Config gốc
base_config_path = '/kaggle/working/mmrotate/configs/oriented_rcnn/oriented-rcnn-le90_r50_fpn_1x_dota.py'
if not os.path.exists(base_config_path):
    print("⚠️ Cảnh báo: Không tìm thấy file config gốc trong /kaggle/working/mmrotate.")
    print("👉 Hãy chắc chắn bạn đã chạy bước CÀI ĐẶT (Clone mmrotate) ở trên.")
else:
    print(f"✅ File config gốc đã sẵn sàng.")

In [2]:
%%writefile custom_ship_config.py

_base_ = 'mmrotate/configs/oriented_rcnn/oriented-rcnn-le90_r50_fpn_1x_dota.py'

# 1. PATHS (Kaggle Standard)
data_root = '/kaggle/input/processed-data/processed_mmrotate_data/'
work_dir = '/kaggle/working/train_output_final'

# 2. CLASS INFO
metainfo = dict(classes=('ship',), palette=[(0, 255, 0)])

# 3. DATALOADER (TỐI ƯU CHO P100)
train_dataloader = dict(
    batch_size=16,      
    num_workers=4,     
    dataset=dict(
        data_root=data_root, metainfo=metainfo,
        ann_file='train/labelTxt/', data_prefix=dict(img_path='train/images/'),
        filter_cfg=dict(filter_empty_gt=True)
    )
)

val_dataloader = dict(
    batch_size=16,      
    num_workers=4,
    dataset=dict(
        data_root=data_root, metainfo=metainfo,
        ann_file='val/labelTxt/', data_prefix=dict(img_path='val/images/'),
        test_mode=True
    )
)

test_dataloader = dict(
    batch_size=16,
    num_workers=4,
    dataset=dict(
        data_root=data_root, metainfo=metainfo,
        ann_file='test/labelTxt/', data_prefix=dict(img_path='test/images/'),
        test_mode=True
    )
)

# 4. EVALUATION
val_evaluator = dict(type='DOTAMetric', metric='mAP')
test_evaluator = val_evaluator

# 5. MODEL HEAD
model = dict(roi_head=dict(bbox_head=dict(num_classes=1)))

# 6. TRAINING SCHEDULE
train_cfg = dict(type='EpochBasedTrainLoop', max_epochs=18, val_interval=1)
val_cfg = dict(type='ValLoop')
test_cfg = dict(type='TestLoop')

# 7. OPTIMIZER & AMP
optim_wrapper = dict(
    type='AmpOptimWrapper', 
    optimizer=dict(
        type='SGD', 
        lr=0.02,        
        momentum=0.9, 
        weight_decay=0.0001
    ),
    # Cấu hình cho Mixed Precision (chống tràn số float16)
    loss_scale='dynamic'
)

# 8. HOOKS
default_hooks = dict(
    checkpoint=dict(type='CheckpointHook', interval=1, save_best='dota/mAP', rule='greater'),
    logger=dict(type='LoggerHook', interval=50),
    visualization=dict(type='mmdet.DetVisualizationHook', draw=True, interval=1)
)

custom_hooks = [
    dict(type='EarlyStoppingHook', monitor='dota/mAP', rule='greater', patience=6, min_delta=0.001)
]

Overwriting custom_ship_config.py


BẮT ĐẦU HUẤN LUYỆN

In [ ]:
import os
from mmengine.config import Config
from mmengine.runner import Runner

checkpoint_file = None

# =================================================================

# 1. Load Config
config_path = '/kaggle/working/custom_ship_config.py'
cfg = Config.fromfile(config_path)

# 2. Xử lý logic Resume
if checkpoint_file and os.path.exists(checkpoint_file):
    print(f"🔄 Đang khôi phục quá trình train từ: {checkpoint_file}")
    print(f"   (Sẽ tiếp tục huấn luyện từ Epoch tiếp theo...)")
    
    cfg.resume = True      
    cfg.load_from = checkpoint_file
else:
    print("🚀 CHẾ ĐỘ: HUẤN LUYỆN MỚI (Start New)")

    cfg.resume = False
    cfg.load_from = None

# 2. Thiết lập Runner
print("\n⚙️ Đang khởi tạo Runner (Model, Dataloader, Optimizer)...")
runner = Runner.from_cfg(cfg)

# 3. Bắt đầu Train
print("🔥 BẮT ĐẦU HUẤN LUYỆN ORIENTED R-CNN...")
print("   (Lưu ý: Log sẽ hiện bên dưới, bạn có thể thấy loss giảm dần theo từng epoch)")
runner.train()

Kiểm tra trên tập Test

In [3]:
import os
import glob
from mmengine.config import Config
from mmengine.runner import Runner

# ==============================================================================
# 1. CẤU HÌNH & TÌM CHECKPOINT
# ==============================================================================

# Đường dẫn file config
config_path = '/kaggle/working/custom_ship_config.py'
cfg = Config.fromfile(config_path)

# Đường dẫn thư mục chứa kết quả train
work_dir = '/kaggle/input/backup-train-2/'

print(f"📂 Đang tìm checkpoint trong: {work_dir}")

# A. Ưu tiên 1: Tìm file 'best_mAP' (File tốt nhất được lưu lại bởi Hook)
best_checkpoints = glob.glob(os.path.join(work_dir, 'best_dota_mAP_*.pth'))

# B. Ưu tiên 2: Nếu không có best, lấy file epoch cuối cùng (vd: epoch_12.pth)
last_checkpoints = glob.glob(os.path.join(work_dir, 'epoch_*.pth'))

if best_checkpoints:
    checkpoint_path = sorted(best_checkpoints, key=os.path.getmtime)[-1]
    print(f"🏆 Đã tìm thấy Checkpoint TỐT NHẤT: {os.path.basename(checkpoint_path)}")
elif last_checkpoints:
    checkpoint_path = sorted(last_checkpoints, key=os.path.getmtime)[-1]
    print(f"⚠️ Không thấy file 'best', sử dụng Checkpoint CUỐI CÙNG: {os.path.basename(checkpoint_path)}")
else:
    raise FileNotFoundError("❌ LỖI: Không tìm thấy bất kỳ file .pth nào! Có thể quá trình Train bị lỗi hoặc chưa chạy xong.")

# ==============================================================================
# 2. CHẠY TEST
# ==============================================================================

# Gán đường dẫn checkpoint vào config
cfg.load_from = checkpoint_path

# Khởi tạo Runner
runner = Runner.from_cfg(cfg)

print("\n🚀 Bắt đầu chạy đánh giá trên tập TEST...")
print("   (Quá trình này sẽ tính toán mAP trên toàn bộ tập dữ liệu kiểm thử)")

# Chạy test
runner.test()

📂 Đang tìm checkpoint trong: /kaggle/input/backup-train-2/
🏆 Đã tìm thấy Checkpoint TỐT NHẤT: best_dota_mAP_epoch_12.pth
12/19 14:26:23 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 387036542
    GPU 0: Tesla P100-PCIE-16GB
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.5, V12.5.82
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
    PyTorch: 2.3.0+cu121
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.3.6 (Git Hash 86e6af5974177e513fd3fee58425e1063e7f1361)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - N

2025-12-19 14:34:39.224861: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766154879.258654     301 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766154879.269108     301 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-19 14:34:39.323895: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-19 14:34:39.328803: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:17661

12/19 14:34:50 - mmengine - INFO - 
+-------+------+------+--------+-------+
| class | gts  | dets | recall | ap    |
+-------+------+------+--------+-------+
| ship  | 3188 | 4232 | 0.904  | 0.863 |
+-------+------+------+--------+-------+
| mAP   |      |      |        | 0.863 |
+-------+------+------+--------+-------+
12/19 14:34:50 - mmengine - INFO - Epoch(test) [44/44]    dota/mAP: 0.8627  dota/AP50: 0.8630  data_time: 5.2794  time: 7.8104


{'dota/mAP': 0.8627297878265381, 'dota/AP50': 0.863}

BÁO CÁO KẾT QUẢ (REPORTING)

Biểu đồ Loss & mAP

In [ ]:
import json
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator # <--- THÊM DÒNG NÀY ĐỂ XỬ LÝ SỐ NGUYÊN
import numpy as np
import os
import glob
import re

# ==============================================================================
# 1. CẤU HÌNH GIAO DIỆN (THEME OCEANEYE)
# ==============================================================================
# --- MÀU SẮC ---
BG_COLOR = '#0a192f'   # Nền xanh đen đậm
TEXT_COLOR = '#ccd6f6' # Chữ màu trắng xám
ACC_COLOR = '#64ffda'  # Màu Cyan (cho mAP)
LOSS_COLOR = '#ff6b6b' # Màu Đỏ cam (cho Loss)
GRID_COLOR = '#112240' # Màu lưới mờ

# Cài đặt global cho Matplotlib
plt.rcParams.update({
    'figure.facecolor': BG_COLOR,
    'axes.facecolor': BG_COLOR,
    'axes.edgecolor': GRID_COLOR,
    'axes.labelcolor': TEXT_COLOR,
    'xtick.color': TEXT_COLOR,
    'ytick.color': TEXT_COLOR,
    'text.color': TEXT_COLOR,
    'grid.color': GRID_COLOR,
    'grid.linestyle': '--',
    'legend.facecolor': BG_COLOR,
    'legend.edgecolor': GRID_COLOR,
    'legend.labelcolor': TEXT_COLOR,
    'savefig.facecolor': BG_COLOR
})

# Đường dẫn thư mục chứa log trên Kaggle
LOG_DIR = '/kaggle/input/' 
OUTPUT_FILE = '/kaggle/working/orn_chart.png'
MODEL_NAME = "Oriented R-CNN (Ship Detection)"

# ==============================================================================
# 2. HÀM XỬ LÝ FILE LOG
# ==============================================================================

def get_dataset_priority(filepath):
    """Sắp xếp file log theo thứ tự train -> train-2 -> train-3"""
    match = re.search(r'train(?:-(\d+))?', filepath)
    if match:
        return int(match.group(1)) if match.group(1) else 1
    return 999

def parse_merged_logs(log_dir):
    print(f"📂 Đang quét log trong: {log_dir}")
    
    # 1. Tìm và sắp xếp file log
    log_files = glob.glob(os.path.join(log_dir, '**', '*.json'), recursive=True)
    log_files = [f for f in log_files if 'config' not in os.path.basename(f)]
    log_files = sorted(log_files, key=get_dataset_priority)

    if not log_files:
        print("❌ LỖI: Không tìm thấy file log nào!")
        return [], [], []

    print(f"🔎 Tìm thấy {len(log_files)} file log. Thứ tự xử lý:")
    for f in log_files:
        print(f"   ↳ [Giai đoạn {get_dataset_priority(f)}]: ...{f[-40:]}")

    # Biến lưu trữ tổng hợp
    epoch_losses = {}
    epoch_maps = {}
    
    for log_path in log_files:
        print(f"   🔄 Đang đọc chi tiết: {os.path.basename(log_path)}")
        try:
            with open(log_path, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    try:
                        log = json.loads(line)
                    except json.JSONDecodeError:
                        continue

                    # --- LẤY EPOCH/STEP ---
                    raw_epoch = log.get('epoch', log.get('step'))
                    if raw_epoch is None: continue
                    
                    # Chuyển ép kiểu về int để đảm bảo dữ liệu đầu vào sạch
                    epoch = int(raw_epoch)

                    # --- 1. LẤY LOSS (Training) ---
                    if 'loss' in log:
                        if epoch not in epoch_losses:
                            epoch_losses[epoch] = []
                        epoch_losses[epoch].append(log['loss'])

                    # --- 2. LẤY mAP (Validation) ---
                    mAP_value = None
                    if 'dota/mAP' in log:
                        mAP_value = log['dota/mAP']
                    elif 'coco/bbox_mAP' in log:
                        mAP_value = log['coco/bbox_mAP']
                    elif 'mAP' in log:
                        mAP_value = log['mAP']

                    if mAP_value is not None:
                        if mAP_value <= 1.0:
                            mAP_value *= 100
                        epoch_maps[epoch] = mAP_value
        except Exception as e:
            print(f"⚠️ Lỗi đọc file {log_path}: {e}")

    # --- TỔNG HỢP DỮ LIỆU CUỐI CÙNG ---
    all_epochs = sorted(list(set(epoch_losses.keys()) | set(epoch_maps.keys())))
    
    final_epochs = []
    final_losses = []
    final_maps = []
    
    last_map = 0 

    for ep in all_epochs:
        avg_loss = None
        if ep in epoch_losses and len(epoch_losses[ep]) > 0:
            avg_loss = sum(epoch_losses[ep]) / len(epoch_losses[ep])
        
        if avg_loss is not None:
            final_epochs.append(ep)
            final_losses.append(avg_loss)
            
            current_map = epoch_maps.get(ep, None)
            if current_map is not None:
                final_maps.append(current_map)
                last_map = current_map
            else:
                final_maps.append(last_map)

    if final_epochs:
        print(f"✅ Đã trích xuất tổng cộng: {len(final_epochs)} Epochs.")
        print(f"   - Loss cuối: {final_losses[-1]:.4f}")
        print(f"   - mAP cuối: {final_maps[-1]:.2f}%")
    else:
        print("⚠️ Không trích xuất được dữ liệu hợp lệ nào.")

    return final_epochs, final_losses, final_maps

# ==============================================================================
# 3. HÀM VẼ BIỂU ĐỒ (VISUALIZATION)
# ==============================================================================
def draw_chart():
    epochs, losses, maps = parse_merged_logs(LOG_DIR)

    if not epochs or len(epochs) == 0:
        print("⚠️ Không có dữ liệu để vẽ biểu đồ.")
        return

    # Thiết lập khung biểu đồ
    fig, ax1 = plt.subplots(figsize=(12, 6))
    
    # --- VẼ LOSS (TRỤC TRÁI) ---
    ax1.set_xlabel('Epochs (Số vòng lặp)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Training Loss', fontsize=12, color=LOSS_COLOR, fontweight='bold')
    
    # Vẽ đường Loss
    line1 = ax1.plot(epochs, losses, color=LOSS_COLOR, marker='o', linewidth=2, 
                     label='Training Loss', markersize=4, alpha=0.8)
    
    ax1.tick_params(axis='y', labelcolor=LOSS_COLOR, colors=LOSS_COLOR)
    ax1.grid(True, linestyle='--', which='major', color=GRID_COLOR, alpha=0.5)

    # ---> QUAN TRỌNG: ÉP TRỤC X CHỈ HIỂN THỊ SỐ NGUYÊN <---
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))

    # --- VẼ mAP (TRỤC PHẢI) ---
    ax2 = ax1.twinx()
    ax2.set_ylabel('mAP Accuracy (%)', fontsize=12, color=ACC_COLOR, fontweight='bold')

    # Vẽ mAP
    line2 = ax2.plot(epochs, maps, color=ACC_COLOR, marker='s', linewidth=2.5, 
                     label='mAP Score', markersize=5)

    ax2.tick_params(axis='y', labelcolor=ACC_COLOR, colors=ACC_COLOR)

    # Chỉnh viền biểu đồ cho đẹp
    for ax in [ax1, ax2]:
        ax.spines['bottom'].set_color(TEXT_COLOR)
        ax.spines['top'].set_visible(False)
        ax.spines['left'].set_color(LOSS_COLOR)
        ax.spines['right'].set_color(ACC_COLOR)

    # --- TIÊU ĐỀ & CHÚ THÍCH ---
    plt.title(f'HIỆU SUẤT HUẤN LUYỆN: {MODEL_NAME}', fontsize=16, fontweight='bold', pad=20, color=TEXT_COLOR)

    # Legend chung
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    leg = ax1.legend(lines, labels, loc='center right', facecolor=BG_COLOR, edgecolor=GRID_COLOR)
    for text in leg.get_texts():
        text.set_color(TEXT_COLOR)

    # Highlight điểm Best mAP
    if len(maps) > 0:
        max_map = max(maps)
        max_idx = maps.index(max_map)
        max_epoch = epochs[max_idx]

        if max_map > 0:
            ax2.annotate(f'Best: {max_map:.2f}%',
                         xy=(max_epoch, max_map),
                         xytext=(max_epoch, max_map - 10 if max_map > 80 else max_map + 10),
                         arrowprops=dict(facecolor=ACC_COLOR, shrink=0.05),
                         color=ACC_COLOR, fontweight='bold', ha='center',
                         bbox=dict(boxstyle="round,pad=0.3", fc=BG_COLOR, ec=ACC_COLOR))

    plt.tight_layout()

    # Lưu file và hiển thị
    plt.savefig(OUTPUT_FILE, dpi=300, facecolor=BG_COLOR)
    print(f"\n🎉 XONG! Ảnh biểu đồ đã lưu tại: {OUTPUT_FILE}")
    plt.show()

if __name__ == "__main__":
    draw_chart()

Biểu đồ Precision - Recall

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
from shapely.geometry import Polygon
import warnings
import torch

# --- CẤU HÌNH CHO KAGGLE & OPENMMLAB ---
try:
    from mmdet.apis import init_detector, inference_detector
    from mmrotate.utils import register_all_modules
    # Đăng ký các module của MMRotate
    register_all_modules(init_default_scope=False)
except ImportError as e:
    print(f"⚠️ Cảnh báo Import: {e}")
    print("👉 Hãy chắc chắn bạn đã cài đặt mmrotate, mmdet, mmcv và mmengine.")

# Tắt các cảnh báo
warnings.filterwarnings("ignore")

# ==============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN 
# ==============================================================================

CONFIG_FILE = '/kaggle/working/custom_ship_config.py' 

# Tìm file checkpoint tốt nhất (.pth)
CHECKPOINT_DIR = '/kaggle/input/backup-train-2/'
checkpoint_list = glob.glob(os.path.join(CHECKPOINT_DIR, 'best_dota_mAP_*.pth'))
if not checkpoint_list:
    checkpoint_list = glob.glob(os.path.join(CHECKPOINT_DIR, 'epoch_*.pth'))

if checkpoint_list:
    # Lấy file mới nhất
    CHECKPOINT_FILE = sorted(checkpoint_list, key=os.path.getmtime)[-1]
    print(f"✅ Đã chọn checkpoint: {CHECKPOINT_FILE}")
else:
    # Fallback nếu bạn upload weights từ dataset
    CHECKPOINT_FILE = '/kaggle/input/backup-train-2/epoch_12.pth' # Sửa dòng này
    print(f"⚠️ Không tìm thấy checkpoint trong working, thử fallback: {CHECKPOINT_FILE}")

# Thư mục dữ liệu Test
TEST_ROOT = '/kaggle/input/processed-data/processed_mmrotate_data/test/'
TEST_IMG_DIR = os.path.join(TEST_ROOT, 'images')
TEST_LABEL_DIR = os.path.join(TEST_ROOT, 'labelTxt')

# File kết quả đầu ra
OUTPUT_FILE = '/kaggle/working/pr_curve_analysis.png'

# Theme màu OceanEye
THEME = {
    'BG': '#0a192f',
    'TEXT': '#ccd6f6',
    'LINE': '#64ffda',
    'GRID': '#112240'
}

# ==============================================================================
# HÀM XỬ LÝ: ĐỌC LABEL DOTA
# ==============================================================================
def parse_dota_label(txt_path):
    """Đọc file labelTxt trả về list các Polygon"""
    polygons = []
    if not os.path.exists(txt_path):
        return []

    with open(txt_path, 'r') as f:
        for line in f:
            data = line.strip().split()
            # DOTA format: x1 y1 x2 y2 x3 y3 x4 y4 classname difficulty
            if len(data) >= 8:
                try:
                    # Lấy 8 tọa độ đầu tiên tạo thành đa giác
                    coords = [float(x) for x in data[:8]]
                    poly = Polygon([(coords[0], coords[1]),
                                    (coords[2], coords[3]),
                                    (coords[4], coords[5]),
                                    (coords[6], coords[7])])
                    if poly.is_valid:
                        polygons.append(poly)
                except:
                    continue
    return polygons

# ==============================================================================
# HÀM XỬ LÝ: TÍNH IOU
# ==============================================================================
def calculate_iou(poly1, poly2):
    """Tính IoU giữa 2 đa giác Shapely"""
    if not poly1.is_valid or not poly2.is_valid: return 0.0
    try:
        inter_area = poly1.intersection(poly2).area
        union_area = poly1.area + poly2.area - inter_area
        if union_area <= 0: return 0.0
        return inter_area / union_area
    except:
        return 0.0

# ==============================================================================
# MAIN: CHẠY ĐÁNH GIÁ & VẼ BIỂU ĐỒ
# ==============================================================================
def main():
    print(f"⏳ Đang khởi tạo Model từ config: {os.path.basename(CONFIG_FILE)}...")
    
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    print(f"   Using Device: {device}")
    
    try:
        model = init_detector(CONFIG_FILE, CHECKPOINT_FILE, device=device)
    except Exception as e:
        print(f"❌ Lỗi khởi tạo Model: {e}")
        return

    print(f"🔄 Đang quét dữ liệu test tại: {TEST_IMG_DIR}")
    if not os.path.exists(TEST_IMG_DIR):
        print(f"❌ LỖI: Đường dẫn ảnh không tồn tại: {TEST_IMG_DIR}")
        return

    img_paths = glob.glob(os.path.join(TEST_IMG_DIR, "*.*"))
    valid_imgs = [p for p in img_paths if p.lower().endswith(('.png', '.jpg', '.jpeg'))]

    if not valid_imgs:
        print("❌ LỖI: Không tìm thấy file ảnh nào!")
        return

    detections = []
    total_ground_truths = 0
    
    print(f"🚀 Bắt đầu đánh giá trên {len(valid_imgs)} ảnh...")

    for i, img_path in enumerate(valid_imgs):
        # 1. Tìm file label tương ứng
        basename = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(TEST_LABEL_DIR, basename + ".txt")

        # 2. Lấy Ground Truth (GT)
        gt_polys = parse_dota_label(label_path)
        total_ground_truths += len(gt_polys)

        # Đánh dấu GT nào đã được phát hiện
        gt_matched = [False] * len(gt_polys)

        # 3. Chạy Model lấy Prediction
        try:
            result = inference_detector(model, img_path)
            
            # Kết quả thường nằm trong pred_instances
            if hasattr(result, 'pred_instances'):
                pred = result.pred_instances
                scores = pred.scores.cpu().numpy()
                bboxes = pred.bboxes.cpu().numpy() # (xc, yc, w, h, a) hoặc (x1, y1, x2, y2, x3, y3, x4, y4)
            else:
                # Fallback cho các version cũ hơn
                # result là list[tensor], lấy phần tử đầu tiên
                scores = result[0][:, -1].cpu().numpy() 
                bboxes = result[0][:, :-1].cpu().numpy()

            # 4. So khớp Prediction với Ground Truth
            for j, score in enumerate(scores):
                # Lọc ngưỡng confidence thấp để tăng tốc
                if score < 0.05: continue 
                
                box = bboxes[j]
                
                # Chuyển đổi box thành Polygon
                # Kiểm tra định dạng box: 5 tham số (Rotated Box) hay 8 tham số (Quadri)
                if len(box) == 5: # (xc, yc, w, h, theta)
                    xc, yc, w, h, a = box
                    rect = ((xc, yc), (w, h), np.degrees(a)) # cv2.boxPoints dùng độ
                    box_points = cv2.boxPoints(rect)
                elif len(box) >= 8: # (x1, y1, ..., x4, y4)
                    box_points = box[:8].reshape(4, 2)
                else:
                    continue # Không xác định được format
                
                pred_poly = Polygon(box_points)
                if not pred_poly.is_valid: continue

                # Tìm GT trùng khớp nhất (Max IoU)
                best_iou = 0
                best_gt_idx = -1

                for k, gt_poly in enumerate(gt_polys):
                    iou = calculate_iou(pred_poly, gt_poly)
                    if iou > best_iou:
                        best_iou = iou
                        best_gt_idx = k

                # IoU Threshold 0.5 (Tiêu chuẩn PASCAL VOC)
                is_tp = False
                if best_iou >= 0.5:
                    if not gt_matched[best_gt_idx]:
                        is_tp = True
                        gt_matched[best_gt_idx] = True # TP: Bắt đúng tàu chưa bị bắt
                    else:
                        is_tp = False # FP: Tàu này đã bị bắt bởi box khác điểm cao hơn
                
                detections.append({'score': score, 'is_tp': is_tp})

        except Exception as e:
            # print(f"⚠️ Lỗi ảnh {basename}: {e}")
            continue

        if (i + 1) % 50 == 0:
            print(f"   Processed {i + 1}/{len(valid_imgs)} images...")

    # ==========================================================================
    # TÍNH TOÁN PRECISION - RECALL
    # ==========================================================================
    if not detections:
        print("⚠️ Không có detection nào! Kiểm tra lại model hoặc threshold.")
        return

    # Sắp xếp theo score giảm dần
    detections.sort(key=lambda x: x['score'], reverse=True)

    tps = np.array([1 if d['is_tp'] else 0 for d in detections])
    fps = np.array([1 if not d['is_tp'] else 0 for d in detections])

    tp_cumsum = np.cumsum(tps)
    fp_cumsum = np.cumsum(fps)

    precisions = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-6)
    recalls = tp_cumsum / (total_ground_truths + 1e-6)

    # Tính Average Precision (AP)
    ap = np.trapz(precisions, recalls)

    print(f"\n📊 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST THỰC TẾ:")
    print(f"   - Tổng số tàu thật (GT): {total_ground_truths}")
    print(f"   - Tổng số box dự đoán:   {len(detections)}")
    print(f"   - Average Precision (AP): {ap:.4f}")

    # ==========================================================================
    # VẼ BIỂU ĐỒ 
    # ==========================================================================
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Áp dụng theme
    fig.patch.set_facecolor(THEME['BG'])
    ax.set_facecolor(THEME['BG'])
    
    # Vẽ đường PR
    ax.plot(recalls, precisions, color=THEME['LINE'], linewidth=3, label=f'Model AP = {ap:.2f}')
    ax.fill_between(recalls, precisions, color=THEME['LINE'], alpha=0.15)

    # Trang trí
    ax.set_title('PRECISION-RECALL CURVE (REAL TEST)', fontsize=16, fontweight='bold', color=THEME['TEXT'], pad=20)
    ax.set_xlabel('Recall (Độ phủ)', fontsize=12, fontweight='bold', color=THEME['TEXT'])
    ax.set_ylabel('Precision (Độ chính xác)', fontsize=12, fontweight='bold', color=THEME['TEXT'])
    
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    
    ax.grid(True, linestyle='--', color=THEME['GRID'], alpha=0.5)
    
    # Chỉnh màu trục
    ax.spines['bottom'].set_color(THEME['TEXT'])
    ax.spines['left'].set_color(THEME['TEXT'])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(colors=THEME['TEXT'])

    # Legend
    leg = ax.legend(loc="lower left", facecolor=THEME['BG'], edgecolor=THEME['TEXT'])
    for text in leg.get_texts(): text.set_color(THEME['TEXT'])

    plt.tight_layout()
    plt.savefig(OUTPUT_FILE, dpi=300, facecolor=THEME['BG'])
    print(f"\n✅ XONG! Ảnh biểu đồ đã lưu tại: {OUTPUT_FILE}")
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
Ma trận nhầm lẫn

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
import seaborn as sns
from shapely.geometry import Polygon
import warnings
import torch

# --- CẤU HÌNH IMPORT ---
try:
    from mmdet.apis import init_detector, inference_detector
    from mmrotate.utils import register_all_modules
    register_all_modules(init_default_scope=False)
except ImportError:
    print("⚠️ Cảnh báo: Chưa cài đặt thư viện mmdet/mmrotate.")

# Tắt cảnh báo
warnings.filterwarnings("ignore")

# ==============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ
# ==============================================================================

# --- ĐƯỜNG DẪN MODEL ---
CONFIG_FILE = '/kaggle/working/custom_ship_config.py'

# Tìm checkpoint tốt nhất
CHECKPOINT_DIR = '/kaggle/input/backup-train-2/'
chk_list = glob.glob(os.path.join(CHECKPOINT_DIR, 'best_dota_mAP_*.pth'))
if not chk_list:
    chk_list = glob.glob(os.path.join(CHECKPOINT_DIR, 'epoch_*.pth'))

if chk_list:
    CHECKPOINT_FILE = sorted(chk_list, key=os.path.getmtime)[-1]
    print(f"✅ Sử dụng Checkpoint: {os.path.basename(CHECKPOINT_FILE)}")
else:
    # Fallback nếu bạn upload weights từ dataset bên ngoài
    CHECKPOINT_FILE = '/kaggle/input/backup-train-2/epoch_12.pth' # Sửa nếu cần
    print(f"⚠️ Dùng fallback checkpoint: {CHECKPOINT_FILE}")

# --- ĐƯỜNG DẪN DỮ LIỆU TEST ---
# Trỏ vào folder chứa ảnh test thật trên Kaggle
TEST_ROOT = '/kaggle/input/processed-data/processed_mmrotate_data/test/'
TEST_IMG_DIR = os.path.join(TEST_ROOT, 'images')
TEST_LABEL_DIR = os.path.join(TEST_ROOT, 'labelTxt')

# File kết quả
OUTPUT_FILE = '/kaggle/working/confusion_matrix_oceaneye.png'

# Ngưỡng đánh giá
IOU_THRESHOLD = 0.5
SCORE_THRESHOLD = 0.3

# --- THEME MÀU OCEANEYE ---
THEME = {
    'BG': '#0a192f',       # Nền
    'TEXT': '#ccd6f6',     # Chữ
    'ACCENT': '#64ffda',   # Điểm nhấn (Cyan)
    'GRID': '#112240',     # Lưới
    'ERROR': '#ff6b6b'     # Màu lỗi/FP
}

# ==============================================================================
# 2. HÀM XỬ LÝ
# ==============================================================================
def parse_dota_label(txt_path):
    polygons = []
    if not os.path.exists(txt_path): return []
    with open(txt_path, 'r') as f:
        for line in f:
            data = line.strip().split()
            if len(data) >= 8:
                try:
                    coords = [float(x) for x in data[:8]]
                    poly = Polygon([(coords[0], coords[1]), (coords[2], coords[3]),
                                    (coords[4], coords[5]), (coords[6], coords[7])])
                    if poly.is_valid: polygons.append(poly)
                except: continue
    return polygons

def calculate_iou(poly1, poly2):
    if not poly1.is_valid or not poly2.is_valid: return 0.0
    try:
        inter = poly1.intersection(poly2).area
        union = poly1.area + poly2.area - inter
        return inter / union if union > 0 else 0.0
    except: return 0.0

# ==============================================================================
# 3. CHƯƠNG TRÌNH CHÍNH
# ==============================================================================
def main():
    print("⏳ Đang khởi tạo Model...")
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    try:
        model = init_detector(CONFIG_FILE, CHECKPOINT_FILE, device=device)
    except Exception as e:
        print(f"❌ Lỗi khởi tạo model: {e}")
        return

    # Lọc ảnh hợp lệ
    if not os.path.exists(TEST_IMG_DIR):
        print(f"❌ Không tìm thấy thư mục ảnh: {TEST_IMG_DIR}")
        return

    img_paths = glob.glob(os.path.join(TEST_IMG_DIR, "*.*"))
    valid_imgs = [p for p in img_paths if p.lower().endswith(('.png', '.jpg', '.jpeg'))]
    

    print(f"🚀 Đang tính toán Confusion Matrix trên {len(valid_imgs)} ảnh...")

    # Biến đếm
    TP = 0  # Đúng là tàu, báo là tàu
    FP = 0  # Nền, báo nhầm là tàu
    FN = 0  # Tàu, nhưng bỏ sót (báo là nền)

    for i, img_path in enumerate(valid_imgs):
        basename = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(TEST_LABEL_DIR, basename + ".txt")

        # 1. Lấy GT
        gt_polys = parse_dota_label(label_path)
        gt_matched = [False] * len(gt_polys)

        # 2. Dự đoán
        try:
            result = inference_detector(model, img_path)
            # Xử lý format mmdet 3.x
            if hasattr(result, 'pred_instances'):
                scores = result.pred_instances.scores.cpu().numpy()
                bboxes = result.pred_instances.bboxes.cpu().numpy()
            else:
                dets = result[0]
                scores = dets[:, 5]
                bboxes = dets[:, :5]
        except: continue

        # Lọc score
        keep_idxs = scores > SCORE_THRESHOLD
        scores = scores[keep_idxs]
        bboxes = bboxes[keep_idxs]

        # 3. So khớp
        for j, score in enumerate(scores):
            # Lấy box dự đoán
            box = bboxes[j]
            if len(box) == 5: # xc, yc, w, h, a
                xc, yc, w, h, a = box
                rect = ((xc, yc), (w, h), np.degrees(a))
                box_points = cv2.boxPoints(rect)
            elif len(box) >= 8:
                box_points = box[:8].reshape(4, 2)
            else: continue
            
            pred_poly = Polygon(box_points)
            if not pred_poly.is_valid: continue

            best_iou = 0
            best_gt_idx = -1

            for k, gt_poly in enumerate(gt_polys):
                iou = calculate_iou(pred_poly, gt_poly)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = k

            if best_iou >= IOU_THRESHOLD:
                if not gt_matched[best_gt_idx]:
                    TP += 1
                    gt_matched[best_gt_idx] = True
                else:
                    FP += 1 # Duplicate -> Tính là FP
            else:
                FP += 1 # Bắt nhầm nền -> FP

        # Số tàu thật không được match -> FN
        FN += gt_matched.count(False)

        if (i + 1) % 50 == 0:
            print(f"   Đã xử lý {i + 1}/{len(valid_imgs)} ảnh...")

    # ==========================================================================
    # VẼ BIỂU ĐỒ
    # ==========================================================================
    
    # Tạo dữ liệu ma trận
    # Hàng 1: Ground Truth là Tàu (Ship) -> [TP, FN]
    # Hàng 2: Ground Truth là Nền (Bg)   -> [FP, 0] (TN không xác định trong Detection)
    matrix_data = np.array([[TP, FN], [FP, 0]])

    # Tính phần trăm cho hàng 1
    row1_sum = TP + FN
    tp_pct = TP / row1_sum if row1_sum > 0 else 0
    fn_pct = FN / row1_sum if row1_sum > 0 else 0

    # Nhãn hiển thị
    annot_labels = np.array([
        [f"TP: {TP}\n({tp_pct:.1%})", f"FN: {FN}\n({fn_pct:.1%})"],
        [f"FP: {FP}\n(Báo giả)", "N/A"] 
    ])

    # Thiết lập Figure
    plt.figure(figsize=(10, 8), facecolor=THEME['BG'])
    ax = plt.gca()
    ax.set_facecolor(THEME['BG'])

    # Vẽ Heatmap (Custom màu cho hợp theme xanh đen)
    sns.heatmap(matrix_data, annot=annot_labels, fmt='', 
                cmap=sns.dark_palette(THEME['ACCENT'], as_cmap=True), # Màu Cyan tối dần
                cbar=False, 
                linewidths=2, linecolor=THEME['BG'],
                xticklabels=['Dự đoán: TÀU', 'Dự đoán: NỀN'],
                yticklabels=['Thực tế: TÀU', 'Thực tế: NỀN'],
                annot_kws={"size": 14, "weight": "bold", "color": "white"})

    # Trang trí lại màu chữ trục
    ax.set_title(f'CONFUSION MATRIX (Ngưỡng IoU={IOU_THRESHOLD})', fontsize=18, fontweight='bold', color=THEME['ACCENT'], pad=20)
    ax.set_ylabel('Ground Truth (Nhãn thật)', fontsize=14, color=THEME['TEXT'], fontweight='bold')
    ax.set_xlabel('Prediction (Mô hình dự đoán)', fontsize=14, color=THEME['TEXT'], fontweight='bold')

    # Chỉnh màu tick labels
    ax.tick_params(axis='x', colors=THEME['TEXT'], labelsize=12)
    ax.tick_params(axis='y', colors=THEME['TEXT'], labelsize=12)

    # Thêm chú thích kết quả tổng hợp
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    stats_text = (f"Precision: {precision:.2f}  |  Recall: {recall:.2f}  |  F1-Score: {f1:.2f}\n"
                  f"Tổng Tàu Thật: {TP+FN}  |  Tổng Báo Động Giả: {FP}")
    
    plt.figtext(0.5, 0.02, stats_text, ha="center", fontsize=12, 
                bbox={"facecolor": THEME['GRID'], "alpha":0.5, "pad":5, "edgecolor": THEME['ACCENT']},
                color=THEME['TEXT'])

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15) # Chừa chỗ cho text ở dưới

    # Lưu và hiển thị
    plt.savefig(OUTPUT_FILE, dpi=300, facecolor=THEME['BG'])
    print(f"\n✅ Đã vẽ xong Confusion Matrix! File lưu tại: {OUTPUT_FILE}")
    print(f"📊 Precision: {precision:.4f}, Recall: {recall:.4f}")
    plt.show()

if __name__ == "__main__":
    main()